# 71장 · Dockerfile과 컨테이너 수명 주기

Dockerfile은 이미지를 만드는 지시와 컨테이너가 시작될 때 실행할 기본 명령을 담습니다. 빌드와 실행 중 어느 때 쓰는 줄인지 구분합니다.

FastAPI 프로세스가 언제 시작되는지 Dockerfile의 RUN과 CMD에서 확인합니다. 준비된 환경에서는 컨테이너 ID로 기존 컨테이너의 재시작과 새 컨테이너 실행을 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## Dockerfile의 RUN과 CMD

Dockerfile의 RUN은 이미지를 만들 때 필요한 명령입니다. CMD는 그 이미지에서 컨테이너를 시작할 때 실행할 기본 명령입니다. EXPOSE는 컨테이너가 사용하는 포트를 설명합니다. 호스트에서 접속할 포트의 공개 설정은 별개입니다.


### 빌드 시점의 RUN과 시작 시점의 CMD

<p>RUN python -m pip install은 이미지를 빌드할 때 의존성을 설치합니다. CMD의 JSON 배열은 컨테이너를 시작할 때 쓸 기본 명령과 인자를 담습니다. 셀에서는 이 배열을 읽으므로 아직 서버가 뜨지는 않습니다.</p>


### 셀 01 · Dockerfile의 CMD

71장 입문 이미지의 Dockerfile에서 마지막 CMD 줄을 가져왔습니다. JSON 배열을 읽어 프로그램과 포트를 나눠 보세요.


In [1]:
import json
directive = 'CMD ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]'
argv = json.loads(directive.removeprefix("CMD "))
print(argv[:4], argv[-1])

['python', '-m', 'uvicorn', 'main:app'] 8000


argv의 '8000'은 문자열이며 uvicorn에 넘길 인자입니다. 지금 한 일은 CMD의 JSON 배열을 읽는 것까지입니다. 서버가 실제로 8000번 포트에서 요청을 받는지는 이 출력으로 알 수 없습니다.


### 과제 02 · 빌드 단계의 RUN

requirements.txt를 설치하는 두 줄의 순서가 바뀌었습니다. 파일을 먼저 이미지 안에 복사하고 설치가 끝난 뒤 셀 01의 CMD를 두도록 세 줄을 배열하세요.


In [2]:
install = "RUN python -m pip install -r requirements.txt"
copy_requirements = "COPY requirements.txt ./"
# copy_requirements, install, directive로 Dockerfile의 세 줄을 구성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>install = &quot;RUN python -m pip install -r requirements.txt&quot;
copy_requirements = &quot;COPY requirements.txt ./&quot;
print(&quot;\n&quot;.join([copy_requirements, install, directive]))</code></pre>
<p>실행 결과</p><pre><code>COPY requirements.txt ./
RUN python -m pip install -r requirements.txt
CMD [&quot;python&quot;, &quot;-m&quot;, &quot;uvicorn&quot;, &quot;main:app&quot;, &quot;--host&quot;, &quot;0.0.0.0&quot;, &quot;--port&quot;, &quot;8000&quot;]</code></pre>
<p>이 조각에서는 설치할 목록을 COPY에서 가져옵니다. RUN을 그보다 앞에 두면 빌드 단계에 목록이 없습니다. CMD는 이미지에 기록되며 컨테이너를 시작할 때 사용합니다. 이 셀에서는 세 줄의 순서만 확인했습니다.</p>
</details>


## 선택 실습 · 컨테이너 중지와 재시작

강사가 Docker 엔진과 수업용 이미지를 미리 준비한 경우에만 아래 선택 셀의 image_id에 제공받은 로컬 이미지 ID를 넣으세요. 새 컨테이너 하나를 포트 공개 없이 만들고 네트워크를 끈 상태에서 확인한 뒤 그 컨테이너만 정리합니다. 기본값은 Docker를 호출하지 않습니다.


### 같은 이미지에서 만든 컨테이너

<p>이미지는 실행 파일과 환경을 담습니다. docker create로 만든 컨테이너에는 별도의 ID가 생기므로 같은 이미지에서 만든 두 컨테이너도 서로 구별됩니다.</p>

<h3>컨테이너 재시작에서 image_id와 cid 읽기</h3>

<p>선택 셀은 로컬 image_id에서 한 번만 컨테이너를 만듭니다. 뒤의 stop과 start는 같은 cid에 적용됩니다. 소스 파일 변경을 담은 이미지로 옮기려면 그 이미지로 새 컨테이너를 만들어야 합니다.</p>


### create·start·stop에서 유지되는 ID

<p>선택 셀은 create 결과를 cid에 저장한 뒤 같은 cid로 start, stop, start를 차례로 호출합니다. 준비된 이미지 ID를 입력하면 각 실행 뒤 inspect로 running, exited, running을 관찰할 수 있습니다.</p>


### 셀 03 · 선택 실습 · 컨테이너 상태 조회

강사가 제공한 이미지 안에서 Python 대기 프로세스를 실행할 수 있어야 합니다. create는 객체만 만들고, start와 stop은 같은 ID를 대상으로 합니다.

여기서는 API 대신 대기 프로세스를 띄워 컨테이너가 실행 상태를 유지하는지 봅니다. 컨테이너가 실행 중이어도 요청에 답하는지는 아직 모릅니다. API를 올렸을 때는 질문을 보내 상태 코드와 응답을 따로 확인하세요.


In [3]:
import subprocess
image_id = ""  # 준비된 수업용 Python 이미지의 로컬 ID
def docker(*args):
    return subprocess.run(["docker", *args], check=True, capture_output=True, text=True, timeout=40).stdout.strip()

if image_id:
    docker("image", "inspect", image_id)
    cid = docker("create", "--pull", "never", "--network", "none", image_id,
                 "python", "-c", "import time; time.sleep(120)")
    try:
        docker("start", cid)
        print(docker("inspect", "--format", "{{.State.Status}}", cid))
        docker("stop", "--time", "1", cid)
        print(docker("inspect", "--format", "{{.State.Status}}", cid))
        docker("start", cid)
        print(docker("inspect", "--format", "{{.State.Status}}", cid))
    finally:
        docker("rm", "-f", cid)
else:
    print("수업용 로컬 이미지 ID가 없어 컨테이너 실습을 건너뜁니다.")

수업용 로컬 이미지 ID가 없어 컨테이너 실습을 건너뜁니다.


기본값이면 이미지가 준비되지 않았다는 안내만 나옵니다. 엔진에서 실행했을 때 확인할 상태는 running, exited, running입니다. 매번 같은 cid를 inspect하므로 ‘새 컨테이너를 하나 더 만들었다’는 설명과 구분할 수 있습니다. 수업에서는 강사가 로컬에 준비한 Python 이미지 ID를 image_id에 넣습니다. 차례로 running, exited, running이 나와야 같은 컨테이너의 정지와 재시작을 관찰한 것입니다.


### 포트 인자와 포트 공개

<p>CMD의 마지막 8000은 컨테이너 안에서 앱이 사용할 포트입니다. EXPOSE는 포트 정보를 표시합니다. 호스트에서도 접속하려면 실행 시 별도로 포트를 연결해야 합니다.</p>


## 돌아보기

앱 소스를 이미지에 복사한 뒤 소스 파일을 수정했다면 새 내용을 담아 이미지를 다시 빌드합니다. 같은 설정·같은 이미지의 프로세스만 다시 띄울 때는 기존 컨테이너를 재시작할 수 있습니다.
